## Этап 0. Изучение специфики Minecraft

CLASSES (17)
'bee', 'chicken', 'cow', 'creeper', 'enderman', 'fox', 'frog', 'ghast',
'goat', 'llama', 'pig', 'sheep', 'skeleton', 'spider', 'turtle', 'wolf', 'zombie'

## Этап 1. Подготовка к выполнению проекта

In [ ]:
'''
!mkdir -p mmdetection/datasets mmdetection/artifacts/{videos,metrics,inference}
!wget -O minecraft_dataset.zip "$(curl -s 'https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key=https://disk.yandex.ru/d/ezpvkg_cdDJnNA' | python3 -c "import sys, json; print(json.load(sys.stdin)['href'])")"
!unzip minecraft_dataset.zip -d ./datasets
!wget -P datasets/minecraft https://code.s3.yandex.net/deep-learning-cv/video.mp4?etag=dbd4425ca062452ae65a33312b5a9300
'''

In [ ]:
'''
python3.10 -m venv practicum_venv # создание окружения 
source practicum_venv/bin/activate # активация


python3.10 -m pip install --upgrade pip setuptools wheel
pip3.10 install torch==2.1.0+cu121 torchvision==0.16.0+cu121 torchaudio==2.1.0+cu121 --index-url https://download.pytorch.org/whl/cu121


python3.10
>>> import torch
>>> torch.__version__
'2.1.0+cu121'
>>> torch.cuda.is_available()
True


pip3.10 install -U openmim
mim install mmengine
mim install "mmcv==2.1.0"

# Мы специально понижаем версию numpy для корректной установки зависимостей mmdetection
pip3.10 install numpy==1.26.4

git clone https://github.com/open-mmlab/mmdetection.git
cd mmdetection
# В mmdetection для установки с флагом -e нам придется понизить версию pip
# -e нужен для сборки в editable mode, это позволит не пересобирать 
# mmdetection после изменений, которые мы внесем
pip3.10 install pip==23.2.1
pip3.10 install -v -e . --no-build-isolation


mim download mmdet --config rtmdet_tiny_8xb32-300e_coco --dest .


python demo/image_demo.py demo/demo.jpg rtmdet_tiny_8xb32-300e_coco.py --weights rtmdet_tiny_8xb32-300e_coco_20220902_112414-78e30dcc.pth --device cuda
'''


In [ ]:
'''
pip3.10 install pyyaml==6.0.1
pip3.10 install opencv-python==4.8.1.78
pip3.10 install seaborn==0.13.2
pip3.10 install ultralytics==8.0.196
pip3.10 install tqdm==4.65.0
pip3.10 install fpdf2==2.7.6
'''


## Этап 2. Исследовательский анализ (EDA) и работа с данными

In [ ]:
#!apt-get install tree -y
#!tree mmdetection/datasets/ -L 2
#!C:\Windows\System32\tree.com datasets /F

In [ ]:
!mkdir datasets\minecraft\annotations
!move datasets\minecraft\train\annotations.json datasets\minecraft\annotations\train_annotations.json
#заодно переименуем в valid
!move datasets\minecraft\valid\annotations.json datasets\minecraft\annotations\valid_annotations.json
!move datasets\minecraft\test\annotations.json datasets\minecraft\annotations\test_annotations.json



In [ ]:
!ls datasets\minecraft\annotations
!dir datasets\minecraft\annotations

### Проверьте, что структура JSON корректна.

In [ ]:
CLASSES = ['bee', 'chicken', 'cow', 'creeper', 'enderman', 'fox', 'frog', 'ghast',
'goat', 'llama', 'pig', 'sheep', 'skeleton', 'spider', 'turtle', 'wolf', 'zombie']

In [ ]:
from scripts import check_ann_files
import json
from pathlib import Path

# Путь к папке с аннотациями
annotations_dir = Path("datasets/minecraft/annotations")
files_to_check = ["train_annotations.json", "valid_annotations.json", "test_annotations.json"]
exp_classes = CLASSES

check_ann_files (annotations_dir, files_to_check, exp_classes)


<span style="color: red;">количество классов не совпадает</span>


### ~~Проверьте, что количество изображений совпадает с количеством аннотаций.~~  
они не обязаны совпадать  
### Проверка соответствия картинок и аннотаций

In [ ]:
import json
from pathlib import Path
from scripts import check_img_ann

base_dir = Path("datasets/minecraft")
annotations_dir = base_dir / "annotations"
splits = ["train", "valid", "test"]

check_img_ann (base_dir, annotations_dir, splits)


### Проанализируйте распределение классов: визуализируйте его и напишите вывод, есть ли дисбаланс.

In [ ]:
from pathlib import Path
from scripts import cls_dispersion

# Пути к данным
annotations_dir = Path("datasets/minecraft/annotations")
splits = ["train", "valid", "test"]


cls_dispersion (annotations_dir, splits)
    

- Классы- призраки во валидации: Классы bee, fox, frog, ghast, goat, llama, turtle имеют объекты в TRAIN, но полностью отсутствуют (0 объектов) в VALID и TEST. Модели научатся их детектировать, но метрики (mAP) по ним вы не узнаете. 
- Аномалия с классом wolf: В TRAIN всего 4 волка (0.10%), зато в VALID их аж 138 (19.80%), а в TEST снова 0. Модель физически не сможет выучить этот класс по 4 примерам, и валидация на нем полностью провалится. 
- Перекос TEST сплита: В TEST выборке класс pig занимает 65.53% всех объектов. Оценка mAP будет отражать только качество детектирования свиней. 
- Пустой класс: Класс minecraft-mobs равен 0 везде. Его нужно удалить из конфигурации (оставить 17 классов вместо 18).

### Визуализируйте один тестовый пример. Отрисуйте одно тестовое изображение с выделенными на нём bounding box'ами и подписанными классами

In [ ]:
from scripts import vis_test
from pathlib import Path


annotations_dir = Path("datasets/minecraft/annotations/")
image_dir = Path("datasets/minecraft/test")


vis_test(annotations_dir, image_dir)
    

In [ ]:

from pathlib import Path
from scripts import img_per_obj

annotations_dir = Path("datasets/minecraft/annotations/")

img_per_obj(annotations_dir)
    

In [ ]:
from dataset_demonstration import visialize_sample

visialize_sample()

## Этап 3. Настройка конфигурации моделей

configs/fcos/fcos_minecraft.py  
mmdet/datasets/coco_minecraft.py

### Перед обучением протестируйте инференс на pretrained-модели FCOS в ноутбуке.

In [ ]:
from test_inference_pt import test_inference_pt

test_inference_pt()

### Сначала подготовьте файл datasets/minecraft/data.yaml, чтобы YOLO работал с данными в корректной форме

datasets/minecraft/data.yaml

### Проверьте инференс на pretrained-модели YOLOv8s. Именно её вы будете дообучать. 

In [ ]:
from test_inference_pt_yolo import test_inference_pt_yolo

test_inference_pt_yolo()

## Этап 4. Обучение моделей

### FCOS/MMdetection

In [2]:
import mmcv
from mmengine.config import Config
from mmengine.runner import Runner

config = 'configs/fcos/fcos_minecraft.py'

cfg = Config.fromfile(config)

# print(cfg.pretty_text) # Выведет весь конфиг для проверки
runner = Runner.from_cfg(cfg)
runner.train()


07/02 17:52:40 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: win32
    Python: 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
    CUDA available: False
    MUSA available: False
    numpy_random_seed: 362798093
    MSVC: n/a, reason: fileno
    PyTorch: 2.0.0+cpu
    PyTorch compiling details: PyTorch built with:
  - C++ Version: 199711
  - MSVC 193431937
  - Intel(R) Math Kernel Library Version 2020.0.2 Product Build 20200624 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.7.3 (Git Hash 6dbeffbae1f23cbbeae17adb7b5b13f1f37c080e)
  - OpenMP 2019
  - LAPACK is enabled (usually provided by MKL)
  - CPU capability usage: AVX2
  - Build settings: BLAS_INFO=mkl, BUILD_TYPE=Release, CXX_COMPILER=C:/actions-runner/_work/pytorch/pytorch/builder/windows/tmp_bin/sccache-cl.exe, CXX_FLAGS=/DWIN32 /D_WINDOWS /GR /EHsc /w /bigobj /FS -DUSE_PTHREADPOOL -DNDEBUG -DUSE_KINETO 

c:\Users\esalmin\dev\dl_training\CV1\mmdetetion_workspace\.venv\lib\site-packages\mmengine\utils\manager.py:113: UserWarning: <class 'mmdet.visualization.local_visualizer.DetLocalVisualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


07/02 17:52:41 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
07/02 17:52:41 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                

FCOS(
  (data_preprocessor): DetDataPreprocessor()
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): ResLayer(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
   

### YOLOv8/Ultralytics

In [ ]:
import torch
from ultralytics import YOLO

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_yolo = YOLO('yolov8s.pt').to(device)

results = model_yolo.train(
        data='datasets/minecraft/data.yaml',
        epochs=50,
        project = 'artifacts/yolo',
        imgsz=(512,512),
        batch=2,
        patience=20,
        seed=42,
        device=device,
        hsv_h=0.015,   # <- Применяем цветовые аугментации
        fliplr=0.5,    # <- Применяем отражение
        mosaic=1.0     # <- Применяем мозаику
        ) 
    